# 01_spending_model — Modelo T ~ X ("sin anemia")

Segundo modelo auxiliar del Double ML: predice el **tratamiento** (`log_gasto_anemia_pan_percapita`) a partir de X (contexto territorial), sin ver nunca `prevalencia_anemia`. Su residuo (`residuo_T`) es la parte del gasto que el territorio no explica — el otro insumo que necesita el causal forest, junto con `residuo_Y` de `01_anemia_model.ipynb`.

In [ ]:
import pandas as pd

df = pd.read_csv("data/clean/merged/causal_model_data.csv")

# filtro defensivo de años (el archivo ya viene 2021-2025)
df = df[df["anio"].between(2021, 2025)].copy()

print(f"Shape tras filtro de años: {df.shape}")
print(f"Años presentes: {sorted(df['anio'].unique())}")
print(f"Distritos únicos: {df['ubigeo'].nunique()}  (debería ser 1,889)")


## 1. Filas que se excluyen del entrenamiento de T

Dos banderas afectan específicamente al tratamiento, y por razones distintas:

- **`t_no_disponible = 1` (146 filas):** no hay `ninos_evaluados` para calcular el gasto per cápita — no hay T que predecir, igual que en `01_anemia_model` con Y (de hecho son las mismas 146 filas).
- **`outlier_administrativo = 1` (93 filas):** el gasto está inflado porque el ubigeo del SIAF identifica la sede de la unidad ejecutora (Lima Cercado, San Isidro, Miraflores, etc.), no necesariamente el distrito donde se ejecutó el gasto real. Esa T no es confiable — se excluye solo aquí, en el modelo de T. Y y X de esas 93 filas siguen intactos y se usan con normalidad en `01_anemia_model`.

Estas dos banderas no se superponen (0 filas con ambas a la vez), así que se excluyen en total 239 filas — el 97.5% de la muestra se queda.

In [ ]:
nulos = df.isnull().sum()
print("Columnas con nulos:")
print(nulos[nulos > 0])
print()
print(f"t_no_disponible=1: {(df['t_no_disponible']==1).sum()}")
print(f"outlier_administrativo=1: {(df['outlier_administrativo']==1).sum()}")
print(f"ambas banderas a la vez: {((df['t_no_disponible']==1)&(df['outlier_administrativo']==1)).sum()}")


In [ ]:
cols_X = [
    "pct_cultivo", "pct_construido", "pct_desnudo", "pct_agua_visible",
    "n_edificios", "area_construida_m2", "confianza_media",
    "area_distrito_km2", "densidad_edificios_km2",
    "elevacion_media", "pendiente_media",
    "pct_agua_permanente", "pct_agua_estacional",
    "altitude", "superficie", "pob_densidad_2020",
]

spending_model_df = df[
    (df["t_no_disponible"] == 0) & (df["outlier_administrativo"] == 0)
].copy()

print(f"Filas excluidas: {len(df) - len(spending_model_df)}")
print(f"Filas para entrenar T~X: {spending_model_df.shape[0]}")
print(f"Distritos únicos en el subset: {spending_model_df['ubigeo'].nunique()} de {df['ubigeo'].nunique()} totales")


In [ ]:
X = spending_model_df[cols_X]
T = spending_model_df["log_gasto_anemia_pan_percapita"]

assert X.isnull().sum().sum() == 0, "X tiene nulos, revisar"
assert T.isnull().sum() == 0, "T tiene nulos, revisar"
assert spending_model_df["anio"].between(2021, 2025).all(), "Hay años fuera de 2021-2025"

print("X y T sin nulos, ventana 2021-2025 confirmada ✅")
print(f"X shape: {X.shape}")
print(f"T shape: {T.shape}")
print()
print("Distribución de T (log_gasto_anemia_pan_percapita):")
print(T.describe())
print(f"\n% de filas con T = 0 (sin gasto PAN ejecutado ese año): {(T == 0).mean():.1%}")


**Nota sobre T:** más de la mitad de las filas tienen `log_gasto_anemia_pan_percapita = 0` — muchos distritos-años simplemente no ejecutaron nada del Programa Articulado Nutricional ese año (T está muy concentrado en cero, con una cola larga de distritos que sí gastaron). Esto no rompe el modelo, pero hay que tenerlo presente: la predicción va a tender a agruparse cerca de 0 para la mayoría de filas, y el causal forest más adelante va a estar comparando, en buena parte, "gastó algo" vs. "no gastó nada", no solo variaciones finas de monto.

## 2. Modelo: LightGBM + Optuna + GroupKFold por distrito

Misma arquitectura que `01_anemia_model.ipynb`, por consistencia y porque ya se validó que funciona bien con esta tabla: LightGBM como regresor, Optuna afinando hiperparámetros (80 intentos) minimizando RMSE, y `GroupKFold` agrupado por `ubigeo` para que ningún distrito aparezca a la vez en train y validación (evita que el modelo memorice el distrito por sus variables casi-constantes entre años).

In [ ]:
import lightgbm as lgb
import optuna
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

groups = spending_model_df["ubigeo"]
N_SPLITS = 5


In [ ]:
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    gkf = GroupKFold(n_splits=N_SPLITS)
    rmses = []
    for train_idx, val_idx in gkf.split(X, T, groups):
        model = lgb.LGBMRegressor(**params, random_state=42)
        model.fit(X.iloc[train_idx], T.iloc[train_idx])
        pred = model.predict(X.iloc[val_idx])
        rmses.append(mean_squared_error(T.iloc[val_idx], pred) ** 0.5)
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_params = study.best_params
print(f"Mejor RMSE (CV): {study.best_value:.4f}")
print(f"Mejores hiperparámetros: {best_params}")


## 3. Cross-fitting honesto: predicción out-of-fold y evaluación

In [ ]:
gkf = GroupKFold(n_splits=N_SPLITS)
oof_pred = np.zeros(len(X))
fold_r2, fold_rmse = [], []
feature_importances = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, T, groups)):
    model = lgb.LGBMRegressor(**best_params, random_state=42, verbosity=-1)
    model.fit(X.iloc[train_idx], T.iloc[train_idx])
    pred = model.predict(X.iloc[val_idx])
    oof_pred[val_idx] = pred

    r2 = r2_score(T.iloc[val_idx], pred)
    rmse = mean_squared_error(T.iloc[val_idx], pred) ** 0.5
    fold_r2.append(r2); fold_rmse.append(rmse)
    feature_importances.append(model.feature_importances_)
    print(f"Fold {fold}: R² = {r2:.3f}  RMSE = {rmse:.4f}  (n_val={len(val_idx)})")

print()
print(f"R² promedio (CV, por distrito nunca visto): {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}")
print(f"RMSE promedio (CV): {np.mean(fold_rmse):.4f}")
print(f"R² global sobre todas las predicciones out-of-fold: {r2_score(T, oof_pred):.3f}")


## 4. Validación: ¿el modelo sobreajustó?

Mismo chequeo que en `01_anemia_model`: comparar R² en train (lo que ya vio) contra R² en validación (distrito nunca visto). Si el train es mucho mejor, hay sobreajuste — aunque, igual que antes, el residuo que realmente se usa después siempre sale de la predicción out-of-fold, nunca de train.

In [ ]:
train_r2 = []
for train_idx, val_idx in gkf.split(X, T, groups):
    model = lgb.LGBMRegressor(**best_params, random_state=42, verbosity=-1)
    model.fit(X.iloc[train_idx], T.iloc[train_idx])
    pred_train = model.predict(X.iloc[train_idx])
    train_r2.append(r2_score(T.iloc[train_idx], pred_train))

print(f"R² promedio en TRAIN (lo que ya vio):             {np.mean(train_r2):.3f}")
print(f"R² promedio en VALIDACIÓN (distrito nunca visto): {np.mean(fold_r2):.3f}")
print(f"Brecha train - validación: {np.mean(train_r2) - np.mean(fold_r2):.3f}")
print()
if (np.mean(train_r2) - np.mean(fold_r2)) > 0.15:
    print("⚠️ Brecha grande: hay señal de sobreajuste, revisar regularización.")
else:
    print("✅ Brecha razonable: el modelo no está memorizando el train.")


## 5. Predicho vs. real

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(T, oof_pred, alpha=0.25, s=10)
lims = [min(T.min(), oof_pred.min()), max(T.max(), oof_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="predicción perfecta")
ax.set_xlabel("log_gasto_anemia_pan_percapita real")
ax.set_ylabel("log_gasto_anemia_pan_percapita predicho (out-of-fold)")
ax.set_title(f"T ~ X — Predicho vs. real (R² = {r2_score(T, oof_pred):.3f})")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Importancia de variables

In [ ]:
importancia = pd.DataFrame(feature_importances, columns=cols_X).mean().sort_values(ascending=False)

print("Importancia promedio de cada variable de X (gain, promedio entre los 5 folds):")
print(importancia)

importancia.plot(kind="barh", figsize=(8, 6), title="Importancia de variables — T ~ X (gasto)")


## 7. Guardar resultados en `data/predictions/`

In [ ]:
spending_model_df["pred_gasto_oof"] = oof_pred
spending_model_df["residuo_T"] = spending_model_df["log_gasto_anemia_pan_percapita"] - spending_model_df["pred_gasto_oof"]

OUTPUT_DIR = Path("data/predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cols_salida = ["ubigeo", "anio", "log_gasto_anemia_pan_percapita", "pred_gasto_oof", "residuo_T"]
spending_model_df[cols_salida].to_csv(OUTPUT_DIR / "residuos_spending_model.csv", index=False)

print(f"Guardado: {OUTPUT_DIR / 'residuos_spending_model.csv'}  ({len(spending_model_df)} filas)")
spending_model_df[cols_salida].head()
